# 2.4 Calculate degassing path using EVo

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🧮 &nbsp; EVo </b> is a thermodynamic magma degassing model.

More information on EVo can be found at https://evo-outgas.readthedocs.io/en/latest/index.html

</div>

This example has been modified from the EVo worked example (https://github.com/pipliggins/EVo/tree/main/examples)

## 2.4.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import evo

import os
os.makedirs("output", exist_ok=True) # creates the output folder

If the following error appear once you've run the cell above

``ModuleNotFoundError: No module 'evo'``

Run the next cell, and then go back to the cell above and rerun the cell above

In [ ]:
# Remove the # at the start of the line below if you need to run it
#!python -m pip install -e .

This notebook was created using EVo: 1.1.0 (at git sha 2487939e18d98292f9b8f3a1f0dea04b707bd89f)

## 2.4.2 Import data

We'll use the same melt inclusion dataset from notebook <a href="1_1_MI_Temperature_Thermobar.ipynb">1.1 Calculate temperature from MI data using Thermobar</a>, where we've already calculated temperatures.

In [ ]:
# import melt inclusion dataset
MI = pd.read_csv("output/wieser2021_w_temperatures.csv")

# if you haven't run Exercise 1, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#results_pvsat_vf = pd.read_csv("answers/wieser2021_w_temperatures.csv")

## 2.4.3 Explore options

The parameters in the next cell configure the run for EVo. Like the other tools, EVo has different options for the volatile solubility laws. Here, we'll use the options appropriate for a basaltic composition, but more details are available in EVo's ReadTheDocs. More information on these options can be found on EVo's GitHub https://github.com/pipliggins/EVo

In [ ]:
# Set up fixed model parameters
model = {
    "COMPOSITION": "basalt", # specify the appropriate solubility functions - other options are `phonolite` or `rhyolite`
    "FIND_SATURATION": True, # If `True`, EVo finds the volatile saturation pressure and decompresses from there
    "GAS_SYS": "cohs", # Volatile system to model: `oh`, `coh`, `soh`, `cohs`, `cohsn`
    "FE_SYSTEM": True, # If `True`, fO2 is buffered by melt Fe2+/Fe3+ exchange
    "FO2_buffer_SET": True, # Toggle option to set the initial fO2 using a value relative to a rock buffer (either IW, FMQ or NNO)
    "FH2_SET": False, # Toggle using hydrogen fugacity as a starting condition on/off
    "WTH2O_SET": True, # Toggle using water content in the melt as a starting condition on/off
    "WTCO2_SET": True, # Toggle using carbon dioxide content in the melt as a starting condition on/off
    "SULFUR_SET": True, # Toggle using sulfur content in the melt as a starting condition on/off
}

## 2.4.4 Initial melt composition

We choose the MI with the highest CO<sub>2</sub> content as the starting point for the degassing calcuations

In [ ]:
# row number with highest CO2 content
row = MI['CO2'].idxmax()

print(f"The sample with the highest CO2 content is {row} with {MI['CO2'][row]} wt%")

We also specify the initial <i>f</i>O<sub>2</sub> (ΔFMQ = 0.2 as before)

In [ ]:
DFMQ = 0.2

And combine this all into a single dictionary to use in the calculations in the following sections

In [ ]:
ini_comp = MI.loc[row].to_dict() # convert to dictionary
ini_comp = {k.split(" ")[0]: v for k, v in ini_comp.items()}
ini_comp.update({
    'DFMQ': DFMQ
})

# print initial composition
ini_comp

We set up the initial conditions for EVo here:

In [ ]:
composition = pd.Series(
    {
        "SIO2": float(ini_comp['SiO2']),
        "TIO2": float(ini_comp['TiO2']),
        "AL2O3": float(ini_comp['Al2O3']),
        "FEO": float(ini_comp['FeO']+ini_comp['Fe2O3']*0.8998),
        "MNO": float(ini_comp['MnO']),
        "MGO": float(ini_comp['MgO']),
        "CAO": float(ini_comp['CaO']),
        "NA2O": float(ini_comp['Na2O']),
        "K2O": float(ini_comp['K2O']),
        "P2O5": float(ini_comp['P2O5']),
    }
)

env = pd.Series(
    model
    | {
        "FO2_buffer": "FMQ", # Rock buffer (`IW`, `FMQ`, `NNO`)
        "FO2_buffer_START": ini_comp['DFMQ'], # ... and log-unit offset
        "T_START": ini_comp['T_C'] + 273.15, # Temperature in Kelvin
        # Volatile inputs as melt wt fractions (EVo uses wt fraction internally)
        "WTH2O_START": ini_comp['H2O'] / 100,
        "WTCO2_START": ini_comp['CO2'] / 100,
        "SULFUR_START": ini_comp['S'] / 1e6,
    }
)

## 2.2.5 Run calculation

In [ ]:
# run calculation
results_degas_evo = evo.run_evo(composition, env)

# save your results
results_degas_evo.to_csv("output/results_degas_evo.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_degas_evo

## 2.2.6 Plotting

And we can plot!

In [ ]:
# if you haven't run this Exercise, you can grab the "answer key" files from here - remove the # at the start of the line below and then run the cell
#results_degas_evo = pd.read_csv("answers/results_degas_evo.csv")

In [ ]:
fig, ((ax1, ax2, ax3),(ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(12,8)) # create figure with 2 rows and 3 column panels

# melt compositions as circles, fluid compositions as diamonds

# EVo results from this exercise in green
df = results_degas_evo
ax1.plot(df['H2O_melt'], df['P'], '-', color='green', linewidth=2, label = "EVo")
ax2.plot(df['CO2_melt']*10000, df['P'], '-', color='green', linewidth=2)
ax3.plot(df['mCO2']*100, df['P'], '-', color='green', linewidth=2)
ax4.plot(df['Stot_melt']*10000, df['P'], '-', color='green', linewidth=2)
ax5.plot(2*df['F']/(2*df['F']+1), df['P'], '-', color='green', linewidth=2)
ax6.plot(100*df['mSO2']/(df['mCO2']+df['mSO2']), df['P'], '-', color='green', linewidth=2)

# label axes
ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax2.set_xlabel('CO$_2$ [m] (ppm)')
ax3.set_xlabel('CO$_2$ [f] (mol%)')
ax4.set_ylabel('P (bars)')
ax4.set_xlabel('S [m] (ppm)')
ax5.set_xlabel('Fe$^{3+}$/Fe$_T$ [m]')
ax6.set_xlabel('SO$_2$/(SO$_2$+CO$_2$) [f] (mol%)')

ax6.set_xlim(0,10) # axes range in SO2/SO2+CO2 panel

# add legend
ax1.legend(frameon=False)

plt.tight_layout()